# 第9节：音频编码原理（AAC/MP3）

本 Notebook 包含三个实验，帮助你理解音频编码的核心原理。

**实验内容：**
1. 比较不同 AAC 码率的波形
2. 比较不同 AAC 码率的频谱
3. 比较不同 AAC 码率的文件大小

## 环境准备

确保已安装 ffmpeg、numpy、matplotlib、scipy。

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import fft, fftfreq
import os

# 检查 ffmpeg 是否可用
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '-version'], capture_output=True, text=True, timeout=10)
        version = result.stdout.split('\n')[0]
        print(f"✓ {cmd} 已安装: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {cmd} 未安装")
        return False
    except subprocess.TimeoutExpired:
        print(f"✗ {cmd} 执行超时")
        return False

check_command('ffmpeg')

## 生成测试素材

生成测试音频并转码为不同码率的 AAC。

In [ ]:
def run_ffmpeg_cmd(cmd, description, timeout=5):
    """执行 ffmpeg 命令并检查结果"""
    print(f"  {description}...")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"  ✗ 失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  ✗ 超时 ({timeout}秒)")
        return False

# 生成测试音频
print("生成测试素材中...")

# 生成原始 WAV
cmd = [
    'ffmpeg', '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
    '-c:a', 'pcm_s16le', '-ar', '44100', '-ac', '2', '-y', 'test.wav'
]
run_ffmpeg_cmd(cmd, "生成原始 WAV")

# 转码为不同码率
for bitrate in ['64k', '128k', '256k']:
    cmd = [
        'ffmpeg', '-i', 'test.wav',
        '-c:a', 'aac', '-b:a', bitrate,
        '-y', f'test_{bitrate}.aac'
    ]
    run_ffmpeg_cmd(cmd, f"转码为 {bitrate} AAC")

# 解码为 WAV 以便比较
for bitrate in ['64k', '128k', '256k']:
    cmd = [
        'ffmpeg', '-i', f'test_{bitrate}.aac',
        '-c:a', 'pcm_s16le',
        '-y', f'test_{bitrate}.wav'
    ]
    run_ffmpeg_cmd(cmd, f"解码 {bitrate} AAC 为 WAV")

print("\n测试素材生成完成！")

## 实验1：比较不同 AAC 码率的波形

**目标**：理解码率对音频质量的影响

In [ ]:
# 绘制波形图
print("=" * 50)
print("波形比较")
print("=" * 50)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 原始音频
rate, data = wavfile.read('test.wav')
axes[0, 0].plot(data[:1000, 0])
axes[0, 0].set_title('Original (WAV)')
axes[0, 0].set_xlabel('Sample')
axes[0, 0].set_ylabel('Amplitude')

# 不同码率
for idx, bitrate in enumerate(['64k', '128k', '256k']):
    row = (idx + 1) // 2
    col = (idx + 1) % 2
    
    rate, data = wavfile.read(f'test_{bitrate}.wav')
    axes[row, col].plot(data[:1000, 0])
    axes[row, col].set_title(f'AAC {bitrate}')
    axes[row, col].set_xlabel('Sample')
    axes[row, col].set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('waveform_comparison.png', dpi=150)
plt.show()

print("波形比较图已保存到 waveform_comparison.png")

## 实验2：比较不同 AAC 码率的频谱

**目标**：理解码率对频谱的影响

In [ ]:
def plot_spectrum(audio_data, sample_rate, title, ax):
    """绘制频谱图"""
    # 计算 FFT
    N = len(audio_data)
    yf = fft(audio_data)
    xf = fftfreq(N, 1 / sample_rate)
    
    # 只取正频率部分
    positive_freq = xf[:N//2]
    magnitude = np.abs(yf[:N//2]) / N
    
    # 转换为 dB
    magnitude_db = 20 * np.log10(magnitude + 1e-10)
    
    ax.plot(positive_freq, magnitude_db)
    ax.set_title(title)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_xlim(0, 20000)
    ax.set_ylim(-80, 0)

# 绘制频谱图
print("=" * 50)
print("频谱比较")
print("=" * 50)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 原始音频
rate, data = wavfile.read('test.wav')
plot_spectrum(data[:4096, 0], rate, 'Original (WAV)', axes[0, 0])

# 不同码率
for idx, bitrate in enumerate(['64k', '128k', '256k']):
    row = (idx + 1) // 2
    col = (idx + 1) % 2
    
    rate, data = wavfile.read(f'test_{bitrate}.wav')
    plot_spectrum(data[:4096, 0], rate, f'AAC {bitrate}', axes[row, col])

plt.tight_layout()
plt.savefig('spectrum_comparison.png', dpi=150)
plt.show()

print("频谱比较图已保存到 spectrum_comparison.png")

## 实验3：比较不同 AAC 码率的文件大小

**目标**：理解码率与文件大小的关系

In [ ]:
# 比较文件大小
print("=" * 50)
print("文件大小比较")
print("=" * 50)

files = ['test.wav', 'test_64k.aac', 'test_128k.aac', 'test_256k.aac']

for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"{f}: {size:.1f} KB")

# 计算压缩比
original_size = os.path.getsize('test.wav')
for bitrate in ['64k', '128k', '256k']:
    compressed_size = os.path.getsize(f'test_{bitrate}.aac')
    ratio = original_size / compressed_size
    print(f"\n{bitrate} 压缩比: {ratio:.1f}x")

## 实验4：比较不同 AAC 码率的听感差异

**目标**：理解码率对听感的影响

In [ ]:
from IPython.display import Audio, display

# 播放不同码率的音频
print("=" * 50)
print("听感比较")
print("=" * 50)

print("\n请播放以下音频，比较听感差异：")

# 原始音频
print("\n1. 原始音频 (WAV):")
display(Audio('test.wav'))

# 不同码率
for bitrate in ['64k', '128k', '256k']:
    print(f"\n2. AAC {bitrate}:")
    display(Audio(f'test_{bitrate}.wav'))

print("\n听感对比要点：")
print("- 64kbps：可能有明显的高频丢失和压缩伪影")
print("- 128kbps：大部分音乐听起来不错，但仔细听可能有细微差异")
print("- 256kbps：与原始音频几乎无法区分")

## 总结

通过本实验，你应该掌握了：

1. **音频编码的基本原理**
   - 频域冗余：能量集中在低频
   - 人耳掩蔽效应：强信号掩蔽弱信号

2. **MDCT 的作用**
   - 将时域转换为频域
   - 无边界效应

3. **码率与音质的关系**
   - 码率越高，音质越好
   - 码率越高，文件越大

4. **文件大小比较**
   - 64kbps 压缩比约 20x
   - 128kbps 压缩比约 10x
   - 256kbps 压缩比约 5x

5. **听感差异**
   - 64kbps：可能有明显的高频丢失和压缩伪影
   - 128kbps：大部分音乐听起来不错
   - 256kbps：与原始音频几乎无法区分